# Tutorial 01 — Getting Started with Korean National Assembly Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyusik-yang/assembly-explorer/blob/main/tutorials/01_getting_started.ipynb)

**Research question**: What does Korean legislative data look like, and how can we collect it programmatically?

This notebook introduces the [열린국회정보 API](https://open.assembly.go.kr) — the Korean National Assembly's open data portal — and shows you how to collect structured legislative data for political science research.

**What you'll learn:**
- How the API is structured and what data is available
- How to retrieve bills, member rosters, and vote tallies
- How to export clean datasets for downstream analysis (Stata, R, Python)

**Coverage:** 16th–22nd Assembly (2000–present). Member-sponsored bills only.

---

## Prerequisites

### 1. Get a free API key
1. Go to [open.assembly.go.kr](https://open.assembly.go.kr)
2. Sign up (무료 회원가입) → 마이페이지 → API 키 발급
3. Copy your key (it looks like `d74b52f3...`)

### 2. Add key to Colab secrets (recommended)
In Colab: left sidebar → 🔑 Secrets → Add `ASSEMBLY_API_KEY` with your key.

Or pass it directly in the cell below.

In [ ]:
# Install dependencies
!pip install httpx pandas plotly -q

In [ ]:
import os

# Load API key from Colab secrets, environment, or paste directly
try:
    from google.colab import userdata
    API_KEY = userdata.get('ASSEMBLY_API_KEY')
except Exception:
    API_KEY = os.getenv('ASSEMBLY_API_KEY', '')

if not API_KEY:
    API_KEY = input('Enter your ASSEMBLY_API_KEY: ').strip()

print('API key loaded:', bool(API_KEY))

## The API Client

We use a thin synchronous wrapper around the 열린국회정보 API.
The key design: every method returns `(rows, total_count)` so you always know
whether there are more pages.

In [ ]:
import httpx

class AssemblyAPI:
    """Synchronous client for the Korean National Assembly Open API."""

    BASE = 'https://open.assembly.go.kr/portal/openapi'
    UNIT_CD = {'22':'100022','21':'100021','20':'100020',
               '19':'100019','18':'100018','17':'100017','16':'100016'}

    def __init__(self, api_key: str):
        self.key = api_key

    def _get(self, endpoint: str, **params) -> tuple[list[dict], int]:
        p = {'KEY': self.key, 'Type': 'json',
             **{k: v for k, v in params.items() if v is not None}}
        r = httpx.get(f'{self.BASE}/{endpoint}', params=p, timeout=30)
        r.raise_for_status()
        body = r.json().get(endpoint, [])
        if not body:
            return [], 0
        head  = body[0].get('head', [])
        total = int(head[0].get('list_total_count', 0)) if head else 0
        code  = (head[1].get('RESULT', {}) if len(head) > 1 else {}).get('CODE', '')
        if code == 'INFO-200':
            return [], 0
        rows = body[1].get('row', []) if len(body) > 1 else []
        return (rows if isinstance(rows, list) else [rows]), total

    def search_bills(self, age, bill_name=None, proposer=None,
                     proc_result=None, committee=None,
                     dt_from=None, dt_to=None, page=1, page_size=100):
        return self._get('nzmimeepazxkubdpn', AGE=age, BILL_NAME=bill_name,
                         PROPOSER=proposer, PROC_RESULT=proc_result,
                         COMMITTEE=committee, STR_DT=dt_from, END_DT=dt_to,
                         pIndex=page, pSize=page_size)

    def get_members(self, age, name=None, party=None, committee=None, page_size=300):
        unit_cd = self.UNIT_CD.get(str(age), f"100{str(age).zfill(3)}")
        return self._get('nwvrqwxyaytdsfvhu', UNIT_CD=unit_cd,
                         HG_NM=name, POLY_NM=party, CMIT_NM=committee,
                         pIndex=1, pSize=page_size)

    def get_vote_results(self, age, bill_name=None, page=1, page_size=50):
        return self._get('ncocpgfiaoituanbr', AGE=age, BILL_NAME=bill_name,
                         pIndex=page, pSize=page_size)

    def get_member_votes(self, bill_id, age, member_name=None, page_size=300):
        return self._get('nojepdqqaweusdfbi', BILL_ID=bill_id, AGE=age,
                         HG_NM=member_name, pIndex=1, pSize=page_size)

    def get_bill_proposers(self, bill_id):
        return self._get('BILLINFOPPSR', BILL_ID=bill_id)

    def get_bill_review(self, age, bill_no=None, page_size=20):
        return self._get('nwbpacrgavhjryiph', AGE=age, BILL_NO=bill_no,
                         pIndex=1, pSize=page_size)

api = AssemblyAPI(API_KEY)
print('Client ready.')

## 1. Searching bills

The core query endpoint. Supports filtering by keyword, proposer, committee, date range, and processing result.

In [ ]:
import pandas as pd

# Search for AI-related bills in the 22nd Assembly
rows, total = api.search_bills(age='22', bill_name='인공지능', page_size=50)
df_ai = pd.DataFrame(rows)

print(f'Found {total} AI-related bills (retrieved {len(df_ai)})')
display(df_ai[['BILL_NO','BILL_NAME','RST_PROPOSER','PROPOSE_DT','PROC_RESULT']].head(10))

In [ ]:
# Processing outcome distribution
import plotly.express as px

outcome_counts = df_ai['PROC_RESULT'].fillna('심사중').value_counts()

fig = px.pie(
    values=outcome_counts.values,
    names=outcome_counts.index,
    title='AI-related bills — 22nd Assembly outcomes',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

## 2. Member rosters

Retrieve all members of an assembly with party, district, committee, and election type.

In [ ]:
# All 22nd Assembly members
members, total_m = api.get_members(age='22', page_size=300)
df_members = pd.DataFrame(members)

print(f'Total members: {total_m}  |  Retrieved: {len(df_members)}')
print(df_members.columns.tolist())

In [ ]:
# Party seat distribution
party_counts = df_members['POLY_NM'].value_counts()

fig = px.bar(
    x=party_counts.index,
    y=party_counts.values,
    title='22nd Assembly — seats by party',
    labels={'x': 'Party', 'y': 'Seats'},
    color=party_counts.index,
    color_discrete_map={
        '더불어민주당': '#004EA2',
        '국민의힘': '#E61E2B',
        '개혁신당': '#FF7210',
        '조국혁신당': '#0095DA',
    },
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Election type breakdown (재선 vs 초선)
if 'REELE_GBN_NM' in df_members.columns:
    term_counts = df_members['REELE_GBN_NM'].value_counts()
    fig = px.bar(x=term_counts.index, y=term_counts.values,
                 title='Members by election term',
                 labels={'x': 'Term', 'y': 'Count'})
    fig.update_layout(showlegend=False)
    fig.show()

## 3. Plenary vote results

Bill-level vote tallies: total yes/no/abstain/absent counts per bill.

In [ ]:
# Recent voted bills — 22nd Assembly
votes, total_v = api.get_vote_results(age='22', page_size=30)
df_votes = pd.DataFrame(votes)

# Coerce counts to numeric
for col in ['YES_TCNT', 'NO_TCNT', 'BLANK_TCNT', 'VOTE_TCNT']:
    if col in df_votes.columns:
        df_votes[col] = pd.to_numeric(df_votes[col], errors='coerce').fillna(0).astype(int)

print(f'Bills with plenary votes: {total_v}  |  Retrieved: {len(df_votes)}')
display(df_votes[['BILL_NAME','PROC_RESULT_CD','YES_TCNT','NO_TCNT','BLANK_TCNT']].head(8))

In [ ]:
import plotly.graph_objects as go

# Stacked bar: yes/no/abstain for the most recent 15 bills
df_plot = df_votes.head(15).copy()
labels = df_plot['BILL_NAME'].str[:25] + '...'

fig = go.Figure()
fig.add_bar(name='Yes 찬성', x=labels, y=df_plot['YES_TCNT'], marker_color='#2ECC71')
fig.add_bar(name='No 반대',  x=labels, y=df_plot['NO_TCNT'],  marker_color='#E74C3C')
fig.add_bar(name='Abstain 기권', x=labels, y=df_plot['BLANK_TCNT'], marker_color='#BDC3C7')
fig.update_layout(barmode='stack', title='Plenary vote breakdown — recent bills',
                  height=450, margin=dict(b=140))
fig.update_xaxes(tickangle=-40)
fig.show()

## 4. Per-member vote records

For a specific bill, retrieve each member's individual vote.

In [ ]:
# Pick the first bill that has a BILL_ID and was actually voted on
sample_bill = df_votes.iloc[0]
bill_id  = sample_bill['BILL_ID']
bill_name = sample_bill['BILL_NAME']

print(f'Bill: {bill_name}')
print(f'ID:   {bill_id}')

# Retrieve all member votes
mvotes, mv_total = api.get_member_votes(bill_id=bill_id, age='22')
df_mv = pd.DataFrame(mvotes)

print(f'\nMember votes: {mv_total}')
display(df_mv[['HG_NM','POLY_NM','ORIG_NM','RESULT_VOTE_MOD']].head(10))

In [ ]:
# Party-level breakdown
if 'POLY_NM' in df_mv.columns:
    breakdown = (
        df_mv.groupby(['POLY_NM', 'RESULT_VOTE_MOD'])
        .size().reset_index(name='count')
    )
    fig = px.bar(
        breakdown, x='POLY_NM', y='count', color='RESULT_VOTE_MOD',
        title=f'Party vote breakdown — {bill_name[:40]}',
        labels={'POLY_NM': 'Party', 'count': 'Members', 'RESULT_VOTE_MOD': 'Vote'},
        color_discrete_map={'찬성':'#2ECC71','반대':'#E74C3C','기권':'#BDC3C7','불참':'#ECF0F1'},
        barmode='group', height=350,
    )
    fig.update_xaxes(tickangle=-30)
    fig.show()

## 5. Bulk data collection with pagination

The API returns at most 100 records per call. For research datasets, auto-paginate.

In [ ]:
def collect_all_bills(api, age, bill_name=None, proposer=None,
                      committee=None, proc_result=None,
                      dt_from=None, dt_to=None, max_records=2000):
    """Auto-paginate search_bills up to max_records."""
    all_rows = []
    page = 1
    PAGE_SIZE = 100
    while len(all_rows) < max_records:
        rows, total = api.search_bills(
            age=age, bill_name=bill_name, proposer=proposer,
            committee=committee, proc_result=proc_result,
            dt_from=dt_from, dt_to=dt_to,
            page=page, page_size=PAGE_SIZE,
        )
        all_rows.extend(rows)
        print(f'  Page {page}: +{len(rows)} ({len(all_rows)}/{total})', end='\r')
        if len(all_rows) >= total or not rows:
            break
        page += 1
    print(f'\nDone: {len(all_rows)} records collected (total available: {total})')
    return all_rows, total

# Example: collect all housing-related bills from the 21st Assembly
# (uncomment to run — takes ~10 seconds)
# housing_bills, total = collect_all_bills(api, age='21', bill_name='주거', max_records=500)
# df_housing = pd.DataFrame(housing_bills)
# df_housing.to_csv('housing_bills_21st.csv', index=False, encoding='utf-8-sig')
# print(df_housing.shape)

## 6. Export for Stata / R

All datasets export as UTF-8 with BOM (required for Stata/Excel to handle Korean text correctly).

In [ ]:
# Save member roster
df_members.to_csv('assembly_22_members.csv', index=False, encoding='utf-8-sig')
print('Saved: assembly_22_members.csv')

# Save AI bills
df_ai.to_csv('assembly_22_ai_bills.csv', index=False, encoding='utf-8-sig')
print('Saved: assembly_22_ai_bills.csv')

# Save per-member votes for the sample bill
df_mv.to_csv(f'votes_{bill_id[:12]}.csv', index=False, encoding='utf-8-sig')
print(f'Saved: votes_{bill_id[:12]}.csv')

# In Colab: download files
try:
    from google.colab import files
    files.download('assembly_22_members.csv')
except Exception:
    pass

---

## Summary

| Endpoint | Method | Key params |
|---|---|---|
| `nzmimeepazxkubdpn` | `search_bills` | age, bill_name, proposer, committee, proc_result, dt_from, dt_to |
| `nwvrqwxyaytdsfvhu` | `get_members` | age (mapped to UNIT_CD), party, committee |
| `ncocpgfiaoituanbr` | `get_vote_results` | age, bill_name |
| `nojepdqqaweusdfbi` | `get_member_votes` | bill_id (from search_bills), age |
| `BILLINFOPPSR`      | `get_bill_proposers` | bill_id |
| `nwbpacrgavhjryiph` | `get_bill_review` | age, bill_no |

**Next steps:**
- [Tutorial 02: Legislator Activity & Voting Patterns](./02_voting_analysis.ipynb)
- [Tutorial 03: Co-sponsorship Network Analysis](./03_network_analysis.ipynb)
- [Live app: Assembly Explorer](https://national-assembly-kr.streamlit.app)